# Task 10: Executive Supply Chain Dashboard

In [2]:
!pip install  pyspark findspark 

### Created the spark Session

In [2]:
from pyspark.sql import SparkSession #type:ignore
from pyspark.sql.functions import *   #type:ignore


spark=SparkSession.builder\
    .appName("Supply_Chain_ETL_Project")\
        .getOrCreate()

In [3]:
from google.colab import drive # type:ignore
drive.mount("/content/drive")

Mounted at /content/drive


### Mradul Drive Path

In [5]:
df_1_transformed = spark.read.parquet("/content/drive/MyDrive/Supply_Chain_ETL_Project/data_lake_layer/silver/transformed_data1.parquet")

In [47]:
df_1_transformed.show()

+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---------------------+--------------+--------------------+--------------------------+-----------------+-----------------+------------------+----------------+------------------+----------------+--------------------+--------------+--------------+
|Shipment_ID|Supplier_ID|     Country|  Product_Type|Monthly_Demand_Tons|Shipment_Volume_Tons|Route_Risk_Score|Historical_Delay_Days|Fuel_Price_USD|Political_Risk_Index|Port_Congestion_Index|Inventory_Days|Supplier_Reliability|Alternative_Supplier_Count|Transit_Time_Days|Delay_Probability|Current_Delay_Days|Freight_Cost_USD|Revenue_Impact_USD|Disruption_Event|Freight_Cost_Per_Ton|Delay_Category|Inventory_Risk|
+-----------+-----------+------------+--------------+-------------------+--------------------+----------------+---------------------+--------------+--------------------+---

# Mradul Sharma

## Supply Chain KPIs

In [28]:
#Total Shipments
total_shipments=df_1_transformed.count()
total_shipments

#Total Revenue Impact
total_revenue_impact = df_1_transformed.agg(sum("Revenue_Impact_USD")).collect()[0][0]
total_revenue_impact

#Average Delay
overall_avg_delay = df_1_transformed.agg(avg("Current_Delay_Days")).collect()[0][0]
overall_avg_delay

#Disruption Percentage
disruption_per = df_1_transformed.agg(
    round((count(when(col("Delay_Category").isin("Medium", "High"), 1)) / total_shipments) * 100, 2)
).collect()[0][0]
disruption_per


print(f"--- Supply Chain KPIs ---")
print(f"Total Shipments: {total_shipments}")
print(f"Total Revenue Impact: ${total_revenue_impact:,.2f}")
print(f"Average Delay: {overall_avg_delay:.2f} Days")
print(f"Disruption Percentage: {disruption_per:.2f}%\n")



--- Supply Chain KPIs ---
Total Shipments: 700
Total Revenue Impact: $444,370,749.72
Average Delay: 12.17 Days
Disruption Percentage: 76.43%



## Supplier KPIs

Nisha Chaudhary

In [35]:
#Read the file
df = spark.read.parquet("/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data1.parquet")

In [57]:
#Most Reliable Supplier
most_reliable_supplier = df_1_transformed.groupBy("Supplier_ID").agg(round(avg("Supplier_Reliability"), 2).alias("Avg_Reliability")) \
.orderBy(col("Avg_Reliability").desc())

most_reliable_supplier.show(1)

+-----------+---------------+
|Supplier_ID|Avg_Reliability|
+-----------+---------------+
|     SUP010|           0.83|
+-----------+---------------+
only showing top 1 row


In [58]:
#Highest Revenue Impact Supplier
highest_revenue_supplier = df_1_transformed.groupBy("Supplier_ID").agg(sum("Revenue_Impact_USD").alias("Total_Revenue_Impact")).orderBy(col("Total_Revenue_Impact").desc())
highest_revenue_supplier.show(1)

+-----------+--------------------+
|Supplier_ID|Total_Revenue_Impact|
+-----------+--------------------+
|     SUP033|       1.705854362E7|
+-----------+--------------------+
only showing top 1 row


In [59]:
#Highest Shipment Volume Supplier
highest_shipment_supplier = df_1_transformed.groupBy("Supplier_ID").agg(sum("Shipment_Volume_Tons").alias("Total_Shipment_Volume")) \
.orderBy(col("Total_Shipment_Volume").desc())

highest_shipment_supplier.show(1)

+-----------+---------------------+
|Supplier_ID|Total_Shipment_Volume|
+-----------+---------------------+
|     SUP033|               212861|
+-----------+---------------------+
only showing top 1 row


# Country KPIs

Nisha Chaudhary

In [75]:
#Highest Congested Port
highest_delay_country = df_1_transformed.groupBy("Country").agg(round(avg("Current_Delay_Days"), 2).alias("Average_Delay")) \
.orderBy(col("Average_Delay").desc()).limit(1)

highest_delay_country.show()

+-------+-------------+
|Country|Average_Delay|
+-------+-------------+
|    UAE|        13.55|
+-------+-------------+



In [76]:
#Highest Freight Cost Country
highest_freight_country = df_1_transformed.groupBy("Country").agg(round(avg("Freight_Cost_USD"), 2).alias("Average_Freight_Cost")) \
.orderBy(col("Average_Freight_Cost").desc()).limit(1)

highest_freight_country.show()

+------------+--------------------+
|     Country|Average_Freight_Cost|
+------------+--------------------+
|Saudi Arabia|           232288.88|
+------------+--------------------+



In [77]:
#Highest Disruption Country
highest_freight_country = df_1_transformed.groupBy("Country").agg(round(avg("Freight_Cost_USD"), 2).alias("Average_Freight_Cost")) \
.orderBy(col("Average_Freight_Cost").desc()).limit(1)

highest_freight_country.show()

+------------+--------------------+
|     Country|Average_Freight_Cost|
+------------+--------------------+
|Saudi Arabia|           232288.88|
+------------+--------------------+



Sakshi Maholiya

## Port KPIs

In [4]:
df_3=spark.read.parquet("/content/drive/.shortcut-targets-by-id/1TV3Q-5w59wgnF6t4DLUHr0Rjlt9Z2Yxg/data_lake_layer/silver/transformed_data3/port_level_kpi")
df_3.show()

+-------+---------------------+------------------+--------------------+--------------------+
|Port_ID|total_shipment_volume|    average_delays|average_freight_cost|total_revenue_impact|
+-------+---------------------+------------------+--------------------+--------------------+
|PORT015|              8124772|12.152751423149905|  214388.59107210586|  6.70390265360001E8|
|PORT008|              9497968| 12.30744071954211|  215597.90252657395|  7.81806708970001E8|
|PORT009|              8848303|11.861377506538798|  212471.36741935503| 7.203474394700009E8|
|PORT020|             10195531|11.973076923076922|  215250.86387692305| 8.278795350900017E8|
|PORT007|              9888308|12.068006182380216|   212736.5304945909| 8.112404923299991E8|
|PORT006|              6973828|12.058165548098435|  213529.19153243827| 5.672542896700007E8|
|PORT017|              9367204|12.154987633965375|   213914.4715416322| 7.697727488599998E8|
|PORT005|              7179841|  12.2518756698821|   212286.4722615225

In [7]:
from pyspark.sql.functions import col

In [8]:
# Highest Congested Port
average_delay=df_3.orderBy(col("average_delays").desc()).show(1, truncate=False)

+-------+---------------------+------------------+--------------------+--------------------+
|Port_ID|total_shipment_volume|average_delays    |average_freight_cost|total_revenue_impact|
+-------+---------------------+------------------+--------------------+--------------------+
|PORT004|7417237              |12.579001019367992|210185.23819571832  |6.21134804860001E8  |
+-------+---------------------+------------------+--------------------+--------------------+
only showing top 1 row


In [9]:
# Highest Revenue Impact Port
highest_revenue=df_3.orderBy(col("total_revenue_impact").desc()).show(1, truncate=False)

+-------+---------------------+------------------+--------------------+--------------------+
|Port_ID|total_shipment_volume|average_delays    |average_freight_cost|total_revenue_impact|
+-------+---------------------+------------------+--------------------+--------------------+
|PORT003|13486443             |11.856082238720731|213164.7836893206   |1.1042033251500003E9|
+-------+---------------------+------------------+--------------------+--------------------+
only showing top 1 row


## Mradul Sharma Loading KPIs In Gold

## Deliverables

## Supply Chain KPIs Load in Gold

In [29]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType
kpi_1_data = [(
    int(total_shipments), 
    float(total_revenue_impact), 
    float(overall_avg_delay), 
    float(disruption_per)
)]

kpi_1_schema = StructType([
    StructField("Total_Shipments", IntegerType(), True),
    StructField("Total_Revenue_Impact_USD", DoubleType(), True),
    StructField("Average_Delay_Days", DoubleType(), True),
    StructField("Disruption_Percentage", DoubleType(), True)
])

df_executive_kpis_1 = spark.createDataFrame(kpi_1_data, schema=kpi_1_schema)
df_executive_kpis_1.show()



+---------------+------------------------+------------------+---------------------+
|Total_Shipments|Total_Revenue_Impact_USD|Average_Delay_Days|Disruption_Percentage|
+---------------+------------------------+------------------+---------------------+
|            700|     4.443707497199998E8|12.174285714285714|                76.43|
+---------------+------------------------+------------------+---------------------+



In [51]:
df_executive_kpis_1.write.mode("overwrite").parquet("/content/drive/MyDrive/Supply_Chain_ETL_Project/data_lake_layer/gold/Supply_Chain_KPIs.parquet")
print("Saved Supply_Chain KPIs to Gold layer.")

Saved Supply_Chain KPIs to Gold layer.


## Supplier KPIs Load in Gold

In [83]:
row_reliable = most_reliable_supplier.first()
row_revenue  = highest_revenue_supplier.first()
row_volume   = highest_shipment_supplier.first()    

best_supplier       = str(row_reliable["Supplier_ID"])
max_rev_supplier    = str(row_revenue["Supplier_ID"])
max_vol_supplier    = str(row_volume["Supplier_ID"])

kpi_2_data = [(
    best_supplier,
    max_rev_supplier,
    max_vol_supplier
)]

kpi_2_schema = StructType([
    StructField("Most_Reliable_Supplier", StringType(), True),
    StructField("Highest_Revenue_Impact_Supplier", StringType(), True),
    StructField("Highest_Shipment_Volume_Supplier", StringType(), True)
])



In [84]:
df_kpi_2_suppliers = spark.createDataFrame(kpi_2_data, schema=kpi_2_schema)
df_kpi_2_suppliers.show()


+----------------------+-------------------------------+--------------------------------+
|Most_Reliable_Supplier|Highest_Revenue_Impact_Supplier|Highest_Shipment_Volume_Supplier|
+----------------------+-------------------------------+--------------------------------+
|                SUP010|                         SUP033|                          SUP033|
+----------------------+-------------------------------+--------------------------------+



In [85]:
df_kpi_2_suppliers.write.mode("append").parquet("/content/drive/MyDrive/Supply_Chain_ETL_Project/data_lake_layer/gold/Supplier_KPIs.parquet")
print("Saved Supplier_KPIs KPIs to Gold layer.")

Saved Supplier_KPIs KPIs to Gold layer.


## Port KPIs Load in Gold

In [30]:
highest_delay_row = df_3.orderBy(col("average_delays").desc()).first()
highest_congested_port = highest_delay_row["Port_ID"] 

top_revenue_row = df_3.orderBy(col("total_revenue_impact").desc()).first()
highest_revenue_port = top_revenue_row["Port_ID"] 

kpi_3_data = [(
    str(highest_congested_port),
    str(highest_revenue_port)
)]

kpi_3_schema = StructType([
    StructField("Highest_Congested_Port", StringType(), True),
    StructField("Highest_Revenue_Impact_Port", StringType(), True)
])

df_port_kpis_3 = spark.createDataFrame(kpi_3_data, schema=kpi_3_schema)

In [31]:
df_port_kpis_3.write.mode("append").parquet("/content/drive/MyDrive/Supply_Chain_ETL_Project/data_lake_layer/gold/Port KPIs.parquet")
print("Saved Port KPIs KPIs to Gold layer.")

Saved Port KPIs KPIs to Gold layer.


## Country KPIs Load in Gold

In [ ]:
row_delay   = highest_delay_country.first()
row_freight = highest_freight_country.first()

highest_disruption_country_df = df_1_transformed.groupBy("Country") \
    .agg(count(F.when(col("Delay_Category").isin("Medium", "High"), 1)).alias("Disruption_Count")) \
    .orderBy(col("Disruption_Count").desc()).limit(1)
    
row_disrupt = highest_disruption_country_df.first()    

max_delay_country = str(row_delay["Country"])
max_freight_country = str(row_freight["Country"])
max_disrupt_country = str(row_disrupt["Country"])


kpi_4_data = [(
    max_delay_country,
    max_freight_country,
    max_disrupt_country
)]

kpi_4_schema = StructType([
    StructField("Highest_Delay_Country", StringType(), True),
    StructField("Highest_Freight_Cost_Country", StringType(), True),
    StructField("Highest_Disruption_Country", StringType(), True)
])

df_kpi_4_countries = spark.createDataFrame(kpi_4_data, schema=kpi_4_schema)

In [ ]:
df_kpi_4_countries.write.mode("append").parquet("/content/drive/MyDrive/Supply_Chain_ETL_Project/data_lake_layer/gold/Country KPIs.parquet")
print("Saved Country KPIs KPIs to Gold layer.")

Saved Country KPIs KPIs to Gold layer.
